# 06 — Pairwise judging with a Kaggle model-proxy judge (Task 7)

**This must run in a Kaggle *benchmark* notebook** (that environment provides free model-proxy access through
`kaggle_benchmarks`). Create it from **https://www.kaggle.com/benchmarks/tasks/new**, then paste/import these cells.
No GPU is needed. Internet **on**.

Judging reads the saved, blind pairwise files only (`outputs/eval/` = greedy, `outputs/eval_rp1.1/` = repetition
penalty 1.1) — nothing is regenerated here. Batches of 5 pairs, >= 5 s between requests, up to 3 attempts per batch,
failures logged, and reruns resume from existing judgments. Expected time: ~25 min per decoding setting.

In [ ]:
REPO_URL = "https://github.com/madgod2004/small-model-alignment-starter.git"
REPO_DIR = "/kaggle/working/small-model-alignment-starter"
import os
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
!git log --oneline -1

## 1. Check the judge model is available through the model proxy

In [ ]:
import kaggle_benchmarks as kbench

JUDGE = "google/gemini-3.6-flash"
available = sorted(kbench.llms)
print("available models:", available)
print("default judge:", getattr(kbench.judge_llm, "name", kbench.judge_llm))
if JUDGE not in available:
    raise SystemExit(f"{JUDGE} is not available here; set JUDGE to one of the models above and record it in the report.")

## 2. Smoke test on 5 pairs (one request) before the full run

In [ ]:
!python scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/dpo_uf_vs_sft_alpaca_pairs.jsonl --output_path outputs/judged/smoke/smoke_judged.jsonl --judge_model {JUDGE} --batch_size 5 --limit 5
!head -c 1500 outputs/judged/smoke/smoke_judged.jsonl

## 3. Judge every pairwise file
Each file is checkpointed after every batch; if the notebook stops, rerunning this cell resumes (`--reuse_judgments`).

In [ ]:
import glob
PAIR_DIRS = {"greedy": "outputs/eval", "rp1.1": "outputs/eval_rp1.1"}
for tag, pair_dir in PAIR_DIRS.items():
    files = sorted(glob.glob(f"{pair_dir}/*_pairs.jsonl"))
    print(f"== {tag}: {len(files)} pair files")
    for pairs_path in files:
        name = os.path.basename(pairs_path).replace("_pairs.jsonl", "")
        out = f"outputs/judged/{tag}/{name}_judged.jsonl"
        !PYTHONUNBUFFERED=1 python scripts/judge_pairwise_kaggle.py --pairs_path {pairs_path} --output_path {out} --judge_model {JUDGE} --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments

## 4. Summary table

In [ ]:
import json
for path in sorted(glob.glob("outputs/judged/*/*_summary.json")):
    if "/smoke/" in path:
        continue
    s = json.load(open(path))
    for criterion, c in s["criteria"].items():
        print(f"{path.split('/')[-2]:7} {os.path.basename(path)[:-13]:42} {criterion:11} "
              f"{c['wins']:>3}W {c['losses']:>3}L {c['ties']:>3}T {c['unknowns']:>2}U {c['failures']:>2}F  "
              f"win% excl ties={c['win_rate_excluding_ties'] and round(100 * c['win_rate_excluding_ties'], 1)}  "
              f"ties-as-half={c['win_rate_ties_as_half'] and round(100 * c['win_rate_ties_as_half'], 1)}  "
              f"picked-A={round(100 * s['judge_picked_A_rate'][criterion], 1)}%")

## 5. Archive

In [ ]:
!rm -rf outputs/judged/smoke
!zip -qr /kaggle/working/judged_outputs.zip outputs/judged
!ls -lh /kaggle/working/judged_outputs.zip